# 🧠 Pelatihan Data HumanWrite AI (Ekstraksi NLP)

Notebook ini dirancang untuk berjalan di **Google Colab**. 
Tujuannya: Mengekstrak statistik gaya penulisan manusia dari 8.000+ dokumen teks (PDF/TXT) dan mengubahnya menjadi **Style Profile JSON** yang akan digunakan oleh LLaMA-3.3.

## 📂 Sumber Dataset Target
Anda harus menyiapkan dataset secara mandiri dan mengunggahnya ke Google Drive Anda.
1. **Akademik**: Garuda, SINTA, Universitas Syiah Kuala, UI, ITB, Google Scholar, IEEE, dll.
2. **Profesional**: Bank Mandiri, Telkom, Kemenkeu, Microsoft docs, Apple docs, dll.
3. **Populer**: Kompas, Tempo, Tirto, BBC News, NYT, dll.
4. **Kreatif**: Pramoedya Ananta Toer, Andrea Hirata, Dee Lestari, George Orwell, Stephen King, dll.

Struktur folder di Google Drive harus seperti ini:
```text
My Drive/Dataset_HumanWrite/
  ├── Akademik/
  │   ├── id/  (File PDF/TXT bahasa Indonesia)
  │   └── en/  (File PDF/TXT bahasa Inggris)
  ├── Profesional/
  ├── Populer/
  └── Kreatif/
```

### 1. Instalasi Dependensi & Persiapan

In [ ]:
!pip install spacy stanza pandas PyPDF2 textstat
!python -m spacy download en_core_web_sm

import stanza
stanza.download('id')  # Download model Bahasa Indonesia

from google.colab import drive
drive.mount('/content/drive')

### 2. Fungsi Ekstraksi Teks (PDF/TXT ke String)

In [ ]:
import os
from PyPDF2 import PdfReader

def extract_text(file_path):
    text = ""
    try:
        if file_path.endswith(".pdf"):
            reader = PdfReader(file_path)
            for page in reader.pages:
                extracted = page.extract_text()
                if extracted:
                    text += extracted + "\n"
        elif file_path.endswith(".txt"):
            with open(file_path, 'r', encoding='utf-8') as f:
                text = f.read()
    except Exception as e:
        print(f"Gagal membaca {file_path}: {e}")
    return text.strip()

def load_corpus(base_path):
    corpus_data = []
    for category in ["Akademik", "Profesional", "Populer", "Kreatif"]:
        for lang in ["id", "en"]:
            folder_path = os.path.join(base_path, category, lang)
            if not os.path.exists(folder_path):
                continue
            
            print(f"Memproses {category} - {lang}...")
            for filename in os.listdir(folder_path):
                if filename.endswith(".txt") or filename.endswith(".pdf"):
                    file_path = os.path.join(folder_path, filename)
                    content = extract_text(file_path)
                    if len(content.split()) > 50:
                        corpus_data.append({"mode": category.lower(), "lang": lang, "text": content})
    
    return corpus_data

# JALANKAN INI UNTUK MEMBACA SEMUA 8000 FILE
# dataset_path = '/content/drive/My Drive/Dataset_HumanWrite'
# all_texts = load_corpus(dataset_path)
# print(f"Total dokumen diproses: {len(all_texts)}")

### 3. Logika Analisis NLP (spaCy & Stanza)

In [ ]:
import spacy
import textstat
import numpy as np

nlp_en = spacy.load("en_core_web_sm")
nlp_id = stanza.Pipeline('id', processors='tokenize,pos,lemma,depparse')

def analyze_text(text, lang):
    metrics = {
        "flesch_reading_ease": textstat.flesch_reading_ease(text),
        "sentences": []
    }
    
    if lang == "en":
        doc = nlp_en(text)
        for sent in doc.sents:
            metrics["sentences"].append(len(sent))
    elif lang == "id":
        doc = nlp_id(text)
        for sent in doc.sentences:
            metrics["sentences"].append(len(sent.words))
            
    if metrics["sentences"]:
        metrics["avg_sentence_length"] = np.mean(metrics["sentences"])
        metrics["sentence_length_std"] = np.std(metrics["sentences"])
    else:
        metrics["avg_sentence_length"] = 0
        metrics["sentence_length_std"] = 0
        
    return metrics


### 4. Proses Training & Export ke JSON

In [ ]:
import pandas as pd
import json

def train_profiles(corpus_data):
    df = pd.DataFrame(corpus_data)
    profiles = {}
    
    for mode in df['mode'].unique():
        mode_data = df[df['mode'] == mode]
        
        avg_len = []
        std_len = []
        flesch = []
        
        print(f"Menganalisis mode: {mode} ({len(mode_data)} dokumen)...")
        for idx, row in mode_data.iterrows():
            m = analyze_text(row['text'], row['lang'])
            avg_len.append(m["avg_sentence_length"])
            std_len.append(m["sentence_length_std"])
            flesch.append(m["flesch_reading_ease"])
            
        profile = {
            "style_mode": mode,
            "description": f"Profil bahasa terlatih untuk {mode}",
            "metrics": {
                "avg_sentence_length": float(np.mean(avg_len)),
                "sentence_length_std": float(np.mean(std_len)),
                "flesch_reading_ease": float(np.mean(flesch)),
            },
            "few_shot_examples": [mode_data.iloc[0]['text'][:500] + "..."] if not mode_data.empty else []
        }
        profiles[f"{mode}_style.json"] = profile
        
    return profiles

# JALANKAN UNTUK TRAINING DAN EXPORT
# final_profiles = train_profiles(all_texts)
# for filename, data in final_profiles.items():
#     with open(f"/content/{filename}", "w") as f:
#         json.dump(data, f, indent=4)
#     print(f"Berhasil menyimpan {filename}")

### ✅ Selesai!
Setelah 4 file JSON (`akademik_style.json`, `profesional_style.json`, dll) berhasil dibuat di Colab (Files di sebelah kiri), **unduh** keempat file tersebut, lalu pindahkan/gantikan ke dalam folder lokal proyek Anda di: 
`humanwrite-backend/data/profiles/`